# Obesity Dataset - Data Exploration

In this notebook we're going to explore the obesity prediction dataset and try to understand what the data looks like and clean it up for analysis.


## Loading the data

First we need to import libraries and load the dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')  # to ignore warnings


: 

In [ ]:
# load the data
df = pd.read_csv("../data/obesity_prediction.csv")
print("Dataset loaded!")
print(f"Shape: {df.shape}")


## Looking at the data

Let's see what the data looks like


In [ ]:
# first few rows
df.head()


In [ ]:
# check the info
df.info()


In [ ]:
# basic statistics
df.describe()


## Checking data quality


In [ ]:
# check for missing values
print("Missing values:")
print(df.isnull().sum())
print(f"\nTotal: {df.isnull().sum().sum()}")  # looks like no missing values, that's good!


In [ ]:
# check duplicates
print(f"Duplicates: {df.duplicated().sum()}")


## Data Cleaning

Now we need to clean the data before analyzing it


In [ ]:
# make a copy so we don't mess up the original
df_clean = df.copy()
print(f"Starting with {df_clean.shape[0]} rows")


### Handle missing values (if any)


In [ ]:
# good news - no missing values in this dataset
print(f"Missing values: {df_clean.isnull().sum().sum()}")
# if there were any, we would fill them with median for numbers and mode for categories


### Remove duplicates


In [ ]:
# check and remove duplicates
duplicates = df_clean.duplicated().sum()
print(f"Duplicates found: {duplicates}")

if duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"Removed {duplicates} duplicate rows")
    print(f"New shape: {df_clean.shape}")
else:
    print("No duplicates found")


### Check if data makes sense


In [ ]:
# checking if the values make sense
print("Checking age, height, weight ranges...")
print(f"Age: {df_clean['Age'].min():.0f} to {df_clean['Age'].max():.0f}")
print(f"Height: {df_clean['Height'].min():.2f}m to {df_clean['Height'].max():.2f}m")
print(f"Weight: {df_clean['Weight'].min():.1f}kg to {df_clean['Weight'].max():.1f}kg")
# looks reasonable!


### Looking for outliers


In [ ]:
# making boxplots to see outliers
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

cols_to_check = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
for idx, col in enumerate(cols_to_check):
    sns.boxplot(data=df_clean, y=col, ax=axes[idx])
    axes[idx].set_title(f'{col}')

plt.tight_layout()
plt.show()
# there are some outliers but they might be valid for obesity data so we'll keep them


### Adding BMI column


In [ ]:
# we think BMI would be useful to have - formula is weight/height^2
df_clean['BMI'] = df_clean['Weight'] / (df_clean['Height'] ** 2)

print("BMI stats:")
print(f"Min: {df_clean['BMI'].min():.2f}")
print(f"Max: {df_clean['BMI'].max():.2f}")
print(f"Mean: {df_clean['BMI'].mean():.2f}")

# visualize BMI
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(df_clean['BMI'], bins=30, color='skyblue', edgecolor='black')
plt.title('BMI Distribution')
plt.xlabel('BMI')

plt.subplot(1, 2, 2)
plt.boxplot(df_clean['BMI'])
plt.title('BMI Boxplot')
plt.ylabel('BMI')
plt.tight_layout()
plt.show()


### Encoding categorical variables


In [ ]:
# need to convert categories to numbers for machine learning later
df_encoded = df_clean.copy()

# converting yes/no to 1/0
binary_cols = ["family_history", "FAVC", "SMOKE", "SCC"]
for col in binary_cols:
    df_encoded[col] = df_encoded[col].map({"no": 0, "yes": 1})

# Gender
df_encoded["Gender"] = df_encoded["Gender"].map({"Female": 0, "Male": 1})

# frequency columns
freq_map = {"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3}
df_encoded["CAEC"] = df_encoded["CAEC"].map(freq_map)
df_encoded["CALC"] = df_encoded["CALC"].map(freq_map)

print("Encoded categorical variables")


In [ ]:
# encoding transportation and obesity levels
mtrans_map = {"Public_Transportation": 0, "Walking": 1, "Automobile": 2, "Motorbike": 3, "Bike": 4}
df_encoded["MTRANS"] = df_encoded["MTRANS"].map(mtrans_map)

obesity_map = {
    "Insufficient_Weight": 0,
    "Normal_Weight": 1,
    "Overweight_Level_I": 2,
    "Overweight_Level_II": 3,
    "Obesity_Type_I": 4,
    "Obesity_Type_II": 5,
    "Obesity_Type_III": 6
}
df_encoded["Obesity"] = df_encoded["Obesity"].map(obesity_map)

print(f"Encoded dataset shape: {df_encoded.shape}")


In [ ]:
# check the cleaned data
print("Cleaned data shape:", df_clean.shape)
df_clean.head()


### Save the cleaned data


In [ ]:
# saving the cleaned data for later use
df_clean.to_csv("../data/obesity_cleaned.csv", index=False)
df_encoded.to_csv("../data/obesity_numeric_cleaned.csv", index=False)
print("Saved cleaned datasets!")


## Data Visualization

Now let's visualize the data to understand patterns


### What categories do we have?


In [ ]:
# looking at categorical columns
for col in ['Gender', 'Obesity', 'MTRANS']:
    print(f"\n{col}:")
    print(df_clean[col].value_counts())


### Obesity distribution


In [ ]:
# plotting obesity distribution
plt.figure(figsize=(10, 5))
df_clean['Obesity'].value_counts().plot(kind='bar', color='skyblue')
plt.title('Obesity Levels Distribution')
plt.xlabel('Obesity Level')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Age and Gender


In [ ]:
# age distribution
plt.hist(df_clean['Age'], bins=20, color='lightblue', edgecolor='black')
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# obesity by gender
sns.countplot(data=df_clean, x='Obesity', hue='Gender')
plt.title('Obesity by Gender')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Height vs Weight


In [ ]:
# scatter plot of height vs weight
plt.scatter(df_clean['Height'], df_clean['Weight'], alpha=0.5)
plt.title('Height vs Weight')
plt.xlabel('Height (m)')
plt.ylabel('Weight (kg)')
plt.show()


### Other interesting patterns


In [ ]:
# does family history matter?
sns.countplot(data=df_clean, x='family_history', hue='Obesity')
plt.title('Family History vs Obesity')
plt.tight_layout()
plt.show()
# seems like it does!


In [ ]:
# physical activity frequency
plt.hist(df_clean['FAF'], bins=15, color='green', alpha=0.7, edgecolor='black')
plt.title('Physical Activity Frequency')
plt.xlabel('Days per week')
plt.ylabel('Count')
plt.show()


### Correlation heatmap


In [ ]:
# correlation heatmap
plt.figure(figsize=(12, 10))
correlation_matrix = df_encoded.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:
# what correlates most with obesity?
obesity_corr = df_encoded.corr()['Obesity'].sort_values(ascending=False)
print("Correlations with Obesity:")
print(obesity_corr)


## Summary

What we learned from exploring this data:


In [ ]:
print("Dataset Overview:")
print(f"- Original records: 2111")
print(f"- After removing duplicates: {df_clean.shape[0]}")
print(f"- Features: {df_clean.shape[1]}")
print(f"- No missing values")
print(f"\nKey findings:")
print("- BMI is highly correlated with obesity (makes sense!)")
print("- Family history seems to have an impact")
print("- Weight is strongly related to obesity levels")
print("- Age distribution is mostly young adults")
